In [2]:
from libraries.inference_training import Configuration, ImageDataset
from libraries.inference_training import initCudaEnvironment, createTransforms
from libraries.inference_training import drawImageAndFeatureMasks
from libraries.inference_training import exportOnnxModel, writeONNXMeta, loadONNX
from libraries.inference_training import trainModel, saveModel, loadModel
from libraries.inference_training import createModelInstance, testInference
from libraries.engine import evaluate
import libraries.utils as utils
import torch
import os
import random

In [3]:
initCudaEnvironment(numCudaDevices=1,
                    visibleCudaDevices="0",
                    clearCudaDeviceCount=False)

# model create function

In [4]:
def createModel(trainDirectory:str, testDirectory:str, modelName:str, epochs:int, labels: list[str], augment_data:bool, model_path:str, model_description="default"):
    """
    :param trainDirectory: path naar training dataset
    :param testDirectory: path naar testing dataset
    :param modelName: naam van model 
    :param epochs: hoeveelheid epochs
    :param labels: list van labels, geef normaal ["parkeerplaatsen"] als er geen andere objecten zijn
    :param augment_data: bepaald of er image transforms gedaan worden, nog niet getest
    :param model_path: path naar save locatie van model
    :param model_description: beschrijft het model in de ONNX als het gesaved is
    :return: 
    """
    config = Configuration()
    print("Device: " + str(config.device))
    config.setIsCrowd(False)
    config.setDatasetPaths(trainPath=trainDirectory, testPath=testDirectory)
    config.setFilePrefix("")
    config.setModelName(modelName)
    config.setInputSizes(inputWidth=250, inputHeight=250)
    config.setInputCellSize(cellSizeM=0.25, minCellSizeM=0.1, maxCellSizeM=0.5)
    config.setVersion(20250121)
    config.setModelInfo(channels=3, numClasses=2+1,  # (1 + background)
                        bboxOverlap=True, bboxPerImage=250, reuseModel=False)
    config.setEpochs(epochs)
    config.setOnnxInfo(producer="Tygron", description=model_description)
    config.addLegendEntry("Background", 0, "#00000000")
    i = 1
    for label in labels:
        config.addLegendEntry(label, i, ["#"+''.join([random.choice('ABCDEF0123456789') for i in range(6)])])
        i += 1
    
    config.setOnnxMetaData(scoreThreshold=0.2,
                           maskThreshold=0.3,
                           strideFraction=0.5)
    
    config.setTensorInfo(tensorName='input_A:RGB_normalized', batchAmount=1)
    if augment_data:
        trainingDataset = ImageDataset(config, True, createTransforms(True))
        testDataset = ImageDataset(config, False, createTransforms(False))
    else:
        trainingDataset = ImageDataset(config, True, createTransforms(False))
        testDataset = ImageDataset(config, False, createTransforms(False))
    
    print("Train Image count: "+str(trainingDataset.__len__()))
    print("Test Image count: "+str(testDataset.__len__()))
    
    if not trainingDataset.validateFiles(False):
        print("Inconsistent training dataset ")
        trainingDataset.validateFiles(True)
    
    if not testDataset.validateFiles(False):
        print("Inconsistent test dataset ")
        testDataset.validateFiles(True)
    
    print("Pytorch model name " + config.getPytorchModelFileName())
    print("Onnx file name " + config.getOnnxFileName())
    
    model = trainModel(config, trainingDataset, testDataset)
    model.eval()
    
    saveModel(config, model, path=model_path+config.getPytorchModelFileName())
    
    exportOnnxModel(config, model, model_path)
    writeONNXMeta(config)
    
    return model, config

# load existing model

In [5]:
def load_model(path:str, config=None):
    """
    :param path: path naar model save location 
    :param config: config als het eerder is ingesteld
    :return: 
    """
    if not config:
        config = Configuration()
    
    model = createModelInstance(config)
    loadModel(config, model, path)
    
    return model, config

In [6]:
def load_default_config():
    config = Configuration()
    config.setIsCrowd(False)
    config.setFilePrefix("")
    config.setInputSizes(inputWidth=250, inputHeight=250)
    config.setInputCellSize(cellSizeM=0.25, minCellSizeM=0.1, maxCellSizeM=0.5)
    config.setVersion(20250121)
    config.setModelInfo(channels=3, numClasses=2+1,  # (1 + background)
                        bboxOverlap=True, bboxPerImage=250, reuseModel=False)
    config.addLegendEntry("Background", 0, "#00000000")
    config.setOnnxMetaData(scoreThreshold=0.2,
                           maskThreshold=0.3,
                           strideFraction=0.5)
    config.setTensorInfo(tensorName='input_A:RGB_normalized', batchAmount=1)
    
    return config

# combo data model

In [6]:
train_directory = "datasets/combo_overlay_sets/train"
test_directory = "datasets/combo_overlay_sets/test"
modelName = "combo_sets_model"
epochs = 25
labels = ["parkeerplaats"]
augment = False
save_path = "models/"
combo_model, config = createModel(train_directory, test_directory, modelName, epochs, labels, augment, save_path)

Device: cuda
Train Image count: 1600
Test Image count: 2400
Pytorch model name combo_sets_model.pt
Onnx file name combo_sets_model.onnx
Detections per image 250
in features mask: 256


C:\Users\Gebruiker\Desktop\homework\deep_learning_in_practice\engine.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=scaler is not None):


Epoch: [0]  [  0/800]  eta: 0:45:10  lr: 0.000011  loss: 3.5884 (3.5884)  loss_classifier: 0.9493 (0.9493)  loss_box_reg: 0.0504 (0.0504)  loss_mask: 1.7442 (1.7442)  loss_objectness: 0.8032 (0.8032)  loss_rpn_box_reg: 0.0412 (0.0412)  time: 3.3879  data: 0.0529  max mem: 1660
Epoch: [0]  [ 10/800]  eta: 0:32:02  lr: 0.000074  loss: 3.3272 (3.5417)  loss_classifier: 0.9344 (0.8900)  loss_box_reg: 0.1772 (0.1988)  loss_mask: 1.5991 (1.5416)  loss_objectness: 0.5126 (0.8526)  loss_rpn_box_reg: 0.0391 (0.0588)  time: 2.4341  data: 0.0544  max mem: 1849
Epoch: [0]  [ 20/800]  eta: 0:31:01  lr: 0.000136  loss: 2.3108 (2.7530)  loss_classifier: 0.6602 (0.7091)  loss_box_reg: 0.1780 (0.1972)  loss_mask: 0.9646 (1.2006)  loss_objectness: 0.3283 (0.5971)  loss_rpn_box_reg: 0.0368 (0.0492)  time: 2.3359  data: 0.0474  max mem: 1849
Epoch: [0]  [ 30/800]  eta: 0:30:32  lr: 0.000199  loss: 1.7840 (2.3904)  loss_classifier: 0.4110 (0.5810)  loss_box_reg: 0.1917 (0.2091)  loss_mask: 0.7322 (1.0411) 

# result testing

In [7]:
config1 = load_default_config()
model, config1 = load_model("models/combo_models/combo_sets_model.pt",config1)

In [25]:
model.eval()
train_directory = "datasets/combo_overlay_sets/train"
test_directory = "datasets/combo_overlay_sets/test" 
config1.setDatasetPaths(trainPath=train_directory, testPath=test_directory)
testDataset = ImageDataset(config1, False, createTransforms(False))
config1.setModelName("combo_overlay_fix")
config1.setOnnxInfo("elijah", "combo overlay fix")
config1.setAutoLimitLabel(True)


0.25


In [18]:
#exportOnnxModel(config1, model, "models/")
print(testDataset.validateFiles())
testDataLoader = torch.utils.data.DataLoader(
    testDataset,
    batch_size=1,
    shuffle=True,
    collate_fn=utils.collate_fn
)
evaluate(model, testDataLoader, device=config1.device)

True
creating index...
index created!
Test:  [  0/800]  eta: 0:20:43  model_time: 1.5059 (1.5059)  evaluator_time: 0.0188 (0.0188)  time: 1.5543  data: 0.0268  max mem: 517
Test:  [100/800]  eta: 0:06:23  model_time: 0.5096 (0.5203)  evaluator_time: 0.0070 (0.0094)  time: 0.5412  data: 0.0197  max mem: 517
Test:  [200/800]  eta: 0:05:26  model_time: 0.5126 (0.5166)  evaluator_time: 0.0070 (0.0088)  time: 0.5409  data: 0.0195  max mem: 517
Test:  [300/800]  eta: 0:04:32  model_time: 0.5146 (0.5163)  evaluator_time: 0.0090 (0.0092)  time: 0.5472  data: 0.0203  max mem: 517
Test:  [400/800]  eta: 0:03:38  model_time: 0.5121 (0.5167)  evaluator_time: 0.0080 (0.0095)  time: 0.5445  data: 0.0183  max mem: 517
Test:  [500/800]  eta: 0:02:43  model_time: 0.5106 (0.5163)  evaluator_time: 0.0080 (0.0093)  time: 0.5470  data: 0.0203  max mem: 517
Test:  [600/800]  eta: 0:01:49  model_time: 0.5156 (0.5168)  evaluator_time: 0.0090 (0.0096)  time: 0.5472  data: 0.0177  max mem: 517
Test:  [700/800] 

# ONNX bug fix attempt

In [8]:
config1 = load_default_config()
model, config1 = load_model("models/combo_models/combo_sets_model.pt", config1)
model.eval()
train_directory = "datasets/combo_overlay_sets/train"
test_directory = "datasets/combo_overlay_sets/test"
config1.setDatasetPaths(trainPath=train_directory, testPath=test_directory)
testDataset = ImageDataset(config1, False, createTransforms(False))
testDataset.validateFiles()
config1.setTensorInfo(tensorName='input_A:RGB_normalized', batchAmount=1)
config1.setOnnxMetaData(scoreThreshold=0.2,
                        maskThreshold=0.3,
                        strideFraction=0.5)
config1.setInputSizes(inputWidth=250, inputHeight=250)
config1.setInputCellSize(cellSizeM=0.25, minCellSizeM=0.1, maxCellSizeM=0.5)
config1.setVersion(2147483647)
config1.setModelInfo(channels=3, numClasses=2+1,  # (1 + background)
                    bboxOverlap=True, bboxPerImage=250, reuseModel=False)
config1.addLegendEntry("parking_place", 1, "#00ffbf")
config1.setEpochs(25)
config1.setOnnxInfo(producer="Tygron", description="private parking space model trained on combo overlay images, 25 epochs")
config1.setIsCrowd(False)

In [15]:
onnxModel = loadONNX(config1)
writeONNXMeta(config1, onnxModel)
print(f"metadata_props={onnxModel.metadata_props}")

metadata_props=[key: "VERSION"
value: "2147483647"
, key: "DESCRIPTION"
value: "private parking space model trained on combo overlay images, 25 epochs"
, key: "PRODUCER"
value: "Tygron"
, key: "MIN_CELL_SIZE_M"
value: "0.25"
, key: "MAX_CELL_SIZE_M"
value: "0.25"
, key: "MASK_THRESHOLD"
value: "0.3"
, key: "SCORE_THRESHOLD"
value: "0.2"
, key: "INFERENCE_MODE"
value: "1"
, key: "BOX_OVERLAP"
value: "1"
, key: "STRIDE_FRACTION"
value: "0.5"
, key: "MAX_GT_INSTANCES"
value: "250"
, key: "LEGEND_NAMES"
value: "Background, parking_place"
, key: "LEGEND_VALUES"
value: "0, 1"
, key: "LEGEND_COLORS"
value: "#00000000, #00ffbf"
]
